In [1]:
%pip install -q transformers torch

import json
import os
import random
import glob
import logging
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TextClassificationPipeline,
)

CARPETA_DATA = 'data'
CARPETA_SALIDA = 'conjunto_noticias'

os.makedirs(CARPETA_SALIDA, exist_ok=True)

archivos_medios = [
    '20minutos.json',
    'ABC.json',
    'elconfidencial.json',
    'lavanguardia.json',
    'noticias_ElDiario.json',
    'okdiario.json',
    'RTVE.json'
]

categorias_objetivo = ['Internacional', 'Nacional', 'Cultura']
n_muestras = 3
noticias_finales = []

for nombre_archivo in archivos_medios:
    ruta_archivo = os.path.join(CARPETA_DATA, nombre_archivo)
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        noticias_medio = json.load(f)
            
    noticias_por_categoria = {cat: [] for cat in categorias_objetivo}
    for noticia in noticias_medio:
        cat = noticia.get("Categoría")
        if cat in categorias_objetivo:
            noticias_por_categoria[cat].append(noticia)
            
    for cat in categorias_objetivo:
        disponibles = noticias_por_categoria[cat]
        cantidad_a_extraer = min(n_muestras, len(disponibles)) 
        seleccionadas = random.sample(disponibles, cantidad_a_extraer)
        noticias_finales.extend(seleccionadas)

ruta_huffpost = os.path.join(CARPETA_DATA, 'noticias_Huffpost.json')
with open(ruta_huffpost, 'r', encoding='utf-8') as f:
    noticias_huffpost = json.load(f)
disponibles_huffpost = [n for n in noticias_huffpost if n.get("Categoría") == "virales"]
noticias_finales.extend(random.sample(disponibles_huffpost, min(10, len(disponibles_huffpost))))

ruta_mediterraneo = os.path.join(CARPETA_DATA, 'noticias_MediterraneoDigital.json')
with open(ruta_mediterraneo, 'r', encoding='utf-8') as f:
    noticias_mediterraneo = json.load(f)
disponibles_mediterraneo = [n for n in noticias_mediterraneo if n.get("Categoría") == "internacional"]
noticias_finales.extend(random.sample(disponibles_mediterraneo, min(10, len(disponibles_mediterraneo))))

archivos_existentes = glob.glob(os.path.join(CARPETA_SALIDA, 'conjunto_noticias_*.json'))
numeros_existentes = []

for archivo in archivos_existentes:
    nombre_base = os.path.basename(archivo)
    try:
        num_str = nombre_base.replace('conjunto_noticias_', '').replace('.json', '')
        numeros_existentes.append(int(num_str))
    except ValueError:
        pass

siguiente_numero = max(numeros_existentes) + 1 if numeros_existentes else 1
nombre_salida = f'conjunto_noticias_{siguiente_numero}.json'
ruta_salida = os.path.join(CARPETA_SALIDA, nombre_salida)

_CLICKBAIT_LABEL = "Clickbait"
log = logging.getLogger(__name__)

def _cargar_pipeline(device: int) -> TextClassificationPipeline:
    tokenizer = AutoTokenizer.from_pretrained("taniwasl/clickbait_es")
    model = AutoModelForSequenceClassification.from_pretrained("taniwasl/clickbait_es")
    return TextClassificationPipeline(
        task="text-classification",
        model=model,
        tokenizer=tokenizer,
        max_length=25,
        truncation=True,
        add_special_tokens=True,
        device=device,
    )

def clasificar_clickbait(noticias_lista, guardar_en=None, batch_size=32, device=-1, verbose=True):
    if not noticias_lista:
        raise ValueError("La lista de noticias está vacía.")

    pipe = _cargar_pipeline(device)
    titulos = [n["Título"] for n in noticias_lista]
    resultados = []
    
    for i in range(0, len(titulos), batch_size):
        preds = pipe(titulos[i : i + batch_size])
        resultados.extend(preds)

    for noticia, pred in zip(noticias_lista, resultados):
        raw = pred["label"]
        score = round(pred["score"], 4)

        noticia["cb"] = (raw == _CLICKBAIT_LABEL)
        noticia["cb_score"] = score
        noticia["cb_label"] = raw

    if verbose:
        cb_total = sum(1 for n in noticias_lista if n["cb"])
        print(f"\n{'─'*50}")
        print(f" Total noticias : {len(noticias_lista)}")
        print(f" Clickbait      : {cb_total} ({100*cb_total/len(noticias_lista):.1f}%)")
        print(f" No clickbait   : {len(noticias_lista)-cb_total} ({100*(len(noticias_lista)-cb_total)/len(noticias_lista):.1f}%)")
        if cb_total:
            print(f"\n Titulares clickbait detectados:")
            for n in noticias_lista:
                if n["cb"]:
                    print(f"  [{n['cb_score']:.2f}] {n['Periódico']} — {n['Título']}")
        print(f"{'─'*50}\n")

    if guardar_en is not None:
        out = Path(guardar_en)
        out.parent.mkdir(parents=True, exist_ok=True)
        with out.open("w", encoding="utf-8") as f:
            json.dump(noticias_lista, f, ensure_ascii=False, indent=4)
        print(f"Resultado guardado en: {out}")

    #return noticias_lista

clasificar_clickbait(noticias_finales, guardar_en=ruta_salida)

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


──────────────────────────────────────────────────
 Total noticias : 83
 Clickbait      : 26 (31.3%)
 No clickbait   : 57 (68.7%)

 Titulares clickbait detectados:
  [1.00] 20minutos — Casi el 70% de la juventud española no está satisfecha con la democracia y más de la mitad cree que "hace falta mano dura"
  [0.96] 20minutos — Laura Pausini, obligada a parar un concierto para usar una máquina de oxígeno
  [1.00] 20minutos — El curioso origen etimológico del término ‘deseo’
  [1.00] El Confidencial — Meloni intentaba marcarse 'un Orbán' y hasta el presidente le ha tenido que parar los pies
  [1.00] El Confidencial — El plan gratis perfecto para familias este puente de mayo en Barcelona: así es la feria de brujas inspirada en una antigua tradición nórdica
  [1.00] El Confidencial — 'Michael': la bochornosa hagiografía del rey del pop
  [0.76] La Vanguardia — El PNV cancela una reunión en la Moncloa por una “falta de respeto” del PSE con Aitor Esteban
  [1.00] La Vanguardia — Las marione